In [122]:
import time
import json
import math
import numpy as np
from itertools import combinations
import chipsplitting as cs
from chipsplitting.hyperfield import (
    HyperfieldHomogeneousLinearSystem as HVLinearSystem,
    HyperfieldVector
)
from chipsplitting import pairing_matrix, print_matrix, PascalForm
import chipsplitting.hyperfield.utils as utils
import multiprocess as mp

In [2]:
# utils
def save_fundamental_models(model, json_file_name):
    with open(json_file_name, 'w') as json_file:
        json.dump([{"support": support, "solution": [x for x in solution] } for support, solution in model], json_file, default=float)
    
    print(f"JSON dump has been written to {json_file_name}")

def load_fundamental_models(json_file_name):
    with open(json_file_name, 'r') as json_file:
        data = json.load(json_file)
        return [(item["support"], item["solution"]) for item in data]

def expand_expression(expression):
    if expression.startswith('t^'):
        power = int(expression[2:])
        return [1 if k == power else 0 for k in range(power + 1)]
    elif expression.startswith('(1-t)^'):
        power = int(expression[len('(1-t)^'):])
        return [math.comb(power,k) * (-1)**(k) for k in range(power + 1)]
    else:
        raise ValueError("Invalid expression")

def multiply_expression(left_exp: list[int], right_exp: list[int]):
    left_power = len(left_exp) - 1
    right_power = len(right_exp) - 1
    degree = left_power + right_power
    t_coefficients = [0] * (degree + 1)
    for a in range(left_power + 1):
        for b in range(right_power + 1):
            t_coefficients[a + b] += left_exp[a] * right_exp[b]
    return t_coefficients

def pad_with_zero(list, desired_length: int):
    if len(list) < desired_length:
        return list + [0] * (desired_length - len(list))
    return list

def create_matrix_from_support(degree: int, support: list[int]):
    support = [utils.to_coordinate(index) for index in support]
    A = np.array([pad_with_zero(multiply_expression(expand_expression(f"t^{col}"), expand_expression(f"(1-t)^{row}")), degree + 1) for col, row in support])
    return A.T

def is_fundamental(support_as_index, degree):
    A = create_matrix_from_support(degree, support_as_index)
    b = np.array([1] + [0] * degree)
    sol, norm, rank, singular_values = np.linalg.lstsq(A, b, rcond=None)

    if rank < len(support_as_index):
        return None

    if len(norm) == 0 or np.isclose(norm[0], 0):
        return sol
    
    return None

def apply_symmetry(config):
    def swap(tuple):
        return (tuple[1], tuple[0])
    def to_array_index(tuple):
        return utils.get_array_index(tuple[0], tuple[1])
    return tuple(sorted([to_array_index(swap(utils.to_coordinate(index))) for index in config]))

### Compute possible supports of fundamental models

In [3]:
def find_positive_supports(pos_support_size, degree_end = None, expand = True):
    degree_end = degree_end if degree_end is not None else 2 * pos_support_size - 3
    # hyperfield variety generated by all pascal equations
    S = {}
    for d in range(1, degree_end + 1):
        start = time.time()
        num_cells = cs.utils.gauss(d+1)
        # positive cells are all cells except the (0,0) cell
        num_pos_cells = num_cells - 1
        # skip if the degree has not enough cells
        if num_pos_cells < pos_support_size:
            continue
            
        base_types = ["diag", "row", "col"]
        A = [PascalForm(d, b, k).to_hyperfield() for b in base_types for k in range(d + 1)]
        linear_system = HVLinearSystem(A)
        

        solutions = linear_system.quick_solve_loop(pos_support_size)
        
        if expand:
            # optimizable
            def expand_solution(sol):
                num_selections = pos_support_size - len(sol)
                domain = [x for x in range(cs.utils.gauss(d + 1)) if x not in sol]
                return [tuple(sorted(sol + c)) for c in combinations(domain, num_selections)]

            expanded = []
            for sol in solutions:
                if len(sol) < pos_support_size:
                    expanded += expand_solution(sol)
                else:
                    expanded.append(sol)
            solutions = list(set(tuple(expanded)))
        
        S[d] = solutions
        print(f"d={d}, #supports={len(solutions)}. Elapsed: {time.time() - start}")
    return S


In [4]:
possible_supports = {}
n_begin = 1
n_end = 7
for pos_support_size in range(n_begin + 1, n_end + 2):
    print(f"n={pos_support_size - 1}")
    possible_supports[pos_support_size] = find_positive_supports(pos_support_size, 12, expand = False)
    #possible_supports[pos_support_size] = find_positive_supports(pos_support_size, mode = "fast", expand = True)
print("Reducing possible solutions by applying symmetry...")

possible_supports_copy = {}

for n, d_to_solutions in possible_supports.items():
    d_to_solutions_copy = {}
    for d, solutions in d_to_solutions.items():
        solutions_copy = set()
        for support in solutions:
            if tuple(sorted(cs.reflect_support(support))) not in solutions_copy:
                solutions_copy.add(support)
        d_to_solutions_copy[d] = list(solutions_copy)
    possible_supports_copy[n] = d_to_solutions_copy

possible_supports = possible_supports_copy

for n, d_to_solutions in possible_supports.items():
    print("")
    print(f"n={n-1}")
    for d, solutions in d_to_solutions.items():
        print(f"d={d}, #supports={len(solutions)}")

n=1
d=1, #supports=1. Elapsed: 0.0006079673767089844
d=2, #supports=0. Elapsed: 0.00042319297790527344
d=3, #supports=0. Elapsed: 0.0007941722869873047
d=4, #supports=0. Elapsed: 0.002077817916870117
d=5, #supports=0. Elapsed: 0.0022270679473876953
d=6, #supports=0. Elapsed: 0.002788066864013672
d=7, #supports=0. Elapsed: 0.002620220184326172
d=8, #supports=0. Elapsed: 0.0035419464111328125
d=9, #supports=0. Elapsed: 0.0037093162536621094
d=10, #supports=0. Elapsed: 0.010324239730834961
d=11, #supports=0. Elapsed: 0.006206035614013672
d=12, #supports=0. Elapsed: 0.005964040756225586
n=2
d=2, #supports=3. Elapsed: 0.00023102760314941406
d=3, #supports=1. Elapsed: 0.00031495094299316406
d=4, #supports=0. Elapsed: 0.00045800209045410156
d=5, #supports=0. Elapsed: 0.0006368160247802734
d=6, #supports=0. Elapsed: 0.0008597373962402344
d=7, #supports=0. Elapsed: 0.0011646747589111328
d=8, #supports=0. Elapsed: 0.0015439987182617188
d=9, #supports=0. Elapsed: 0.0018901824951171875
d=10, #supp

In [125]:
def is_positive(solution):
    for x in solution:
        if np.isclose(x,0):
            return False
        if x < 0:
            return False
    return True

def is_parametric(solution):
    for x in solution:
        if not x.is_number:
            return True
    return False

def is_asymmetric(support):
    return list(sorted(support)) != list(sorted(apply_symmetry(support)))

def filter_fundamental_models(fundamental_models):
    return [(support, solution) for support, solution in fundamental_models if solution and is_positive(solution)]

def find_fundamental_models(n, degrees):
    if isinstance(degrees, int):
        degrees = [degrees]
        
    fundamental_models = []
    
    for d in degrees:
        start = time.time()
        supports_to_check = possible_supports[n + 1][d]

        print("")
        print(f"d={d}: {len(supports_to_check)} supports to check")

        for index, support in enumerate(supports_to_check):
            if index == 100:
                elapsed = time.time() - start
                print(f"Solved linear equations of 100 supports in {round(elapsed, 2)}s. Estimate: {round(len(supports_to_check) / 100 * elapsed / 60, 2)}min")

            S = []
            if len(support) <= n:
                num_selections = n + 1 - len(support)
                domain = [x for x in range(cs.utils.gauss(d + 1)) if x not in support]
                S = [tuple(sorted(support + c)) for c in combinations(domain, num_selections)]
            elif len(support) == n + 1:
                S = [support]
            else:
                print("HERE")

            for s in S:
                solution = is_fundamental(s, d)
                fundamental_models.append((s, solution))

        end = time.time()
        print(f"Solved linear equations for all supports. Elapsed time: {round(end - start, 2)}s")

    print()

    
    
    fundamental_models = [(support, solution) for support, solution in fundamental_models if solution is not None]
    print(f"After excluding supports with no solutions, {len(fundamental_models)} fundamental models are left.")

    #fundamental_models = [(support, solution) for support, solution in fundamental_models if not is_parametric(solution)]
    #print(f"After excluding supports with parametric solutions, {len(fundamental_models)} fundamental models are left.")

    fundamental_models = [(support, solution) for support, solution in fundamental_models if is_positive(solution)]
    print(f"After excluding supports with non-positive solutions, {len(fundamental_models)} fundamental models are left.")

    print(f"")
    print(f"We have found {len(fundamental_models)} fundamental models.")

    missing_models_from_symmetry = [(sorted(apply_symmetry(support)), solution) for support, solution in fundamental_models if is_asymmetric(support)]
    print(f"Found additional {len(missing_models_from_symmetry)} fundamental models due to symmetry.")
    
    return list(fundamental_models + missing_models_from_symmetry)




def process_chunk(chunk, n, d):
    """
    Processes a chunk of supports and returns the fundamental models found.
    """
    fundamental_models = []
    for support in chunk:
        S = []
        if len(support) <= n:
            num_selections = n + 1 - len(support)
            domain = [x for x in range(cs.utils.gauss(d + 1)) if x not in support]
            S = [tuple(sorted(support + c)) for c in combinations(domain, num_selections)]
        elif len(support) == n + 1:
            S = [support]
        else:
            print("HERE")

        for s in S:
            solution = is_fundamental(s, d)
            if solution is not None:
                fundamental_models.append((s, solution))
    return fundamental_models


def find_fundamental_models_core(n, degrees):
    if isinstance(degrees, int):
        degrees = [degrees]

    fundamental_models = []

    for d in degrees:
        start = time.time()
        supports_to_check = possible_supports[n + 1][d]

        print("")
        print(f"d={d}: {len(supports_to_check)} supports to check")

        # Split supports into chunks for parallel processing
        num_cores = mp.cpu_count()
        chunk_size = len(supports_to_check) // num_cores
        chunks = [supports_to_check[i:i + chunk_size] for i in range(0, len(supports_to_check), chunk_size)]

        # Use multiprocessing to process chunks
        with mp.Pool(processes=num_cores) as pool:
            results = pool.starmap(process_chunk, [(chunk, n, d) for chunk in chunks])

        # Combine results
        for result in results:
            fundamental_models.extend(result)

        end = time.time()
        print(f"Solved linear equations for all supports. Elapsed time: {round(end - start, 2)}s")

    print()

    fundamental_models = [(support, solution) for support, solution in fundamental_models if solution is not None]
    print(f"After excluding supports with no solutions, {len(fundamental_models)} fundamental models are left.")

    #fundamental_models = [(support, solution) for support, solution in fundamental_models if not is_parametric(solution)]
    #print(f"After excluding supports with parametric solutions, {len(fundamental_models)} fundamental models are left.")

    fundamental_models = [(support, solution) for support, solution in fundamental_models if is_positive(solution)]
    print(f"After excluding supports with non-positive solutions, {len(fundamental_models)} fundamental models are left.")

    print(f"")
    print(f"We have found {len(fundamental_models)} fundamental models.")

    missing_models_from_symmetry = [(sorted(apply_symmetry(support)), solution) for support, solution in fundamental_models if is_asymmetric(support)]
    print(f"Found additional {len(missing_models_from_symmetry)} fundamental models due to symmetry.")
    
    return list(fundamental_models + missing_models_from_symmetry)

## $n = 3$

In [108]:
fundamental_models = find_fundamental_models(2, [2])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=2: 2 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 2 fundamental models are left.
After excluding supports with non-positive solutions, 2 fundamental models are left.

We have found 2 fundamental models.
Found additional 1 fundamental models due to symmetry.

There exist exactly 3 unique fundamental models ✨


In [109]:
fundamental_models = find_fundamental_models(2, [3])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=3: 1 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 1 fundamental models are left.
After excluding supports with non-positive solutions, 1 fundamental models are left.

We have found 1 fundamental models.
Found additional 0 fundamental models due to symmetry.

There exist exactly 1 unique fundamental models ✨


## $n = 4$

In [110]:
fundamental_models = find_fundamental_models(3, [3])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=3: 10 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 15 fundamental models are left.
After excluding supports with non-positive solutions, 7 fundamental models are left.

We have found 7 fundamental models.
Found additional 5 fundamental models due to symmetry.

There exist exactly 12 unique fundamental models ✨


In [111]:
fundamental_models = find_fundamental_models(3, [4])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=4: 8 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 2 fundamental models are left.
After excluding supports with non-positive solutions, 2 fundamental models are left.

We have found 2 fundamental models.
Found additional 2 fundamental models due to symmetry.

There exist exactly 4 unique fundamental models ✨


In [112]:
fundamental_models = find_fundamental_models(3, [5])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=5: 4 supports to check
Solved linear equations for all supports. Elapsed time: 0.0s

After excluding supports with no solutions, 1 fundamental models are left.
After excluding supports with non-positive solutions, 1 fundamental models are left.

We have found 1 fundamental models.
Found additional 1 fundamental models due to symmetry.

There exist exactly 2 unique fundamental models ✨


## $n = 5$

In [113]:
fundamental_models = find_fundamental_models(4, [4])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=4: 81 supports to check
Solved linear equations for all supports. Elapsed time: 0.01s

After excluding supports with no solutions, 146 fundamental models are left.
After excluding supports with non-positive solutions, 53 fundamental models are left.

We have found 53 fundamental models.
Found additional 48 fundamental models due to symmetry.

There exist exactly 82 unique fundamental models ✨


In [114]:
fundamental_models = find_fundamental_models(4, [5])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=5: 198 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.03s

After excluding supports with no solutions, 66 fundamental models are left.
After excluding supports with non-positive solutions, 24 fundamental models are left.

We have found 24 fundamental models.
Found additional 15 fundamental models due to symmetry.

There exist exactly 38 unique fundamental models ✨


In [115]:
fundamental_models = find_fundamental_models(4, [6])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=6: 304 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.04s

After excluding supports with no solutions, 10 fundamental models are left.
After excluding supports with non-positive solutions, 5 fundamental models are left.

We have found 5 fundamental models.
Found additional 5 fundamental models due to symmetry.

There exist exactly 10 unique fundamental models ✨


In [116]:
fundamental_models = find_fundamental_models(4, [7])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=7: 376 supports to check
Solved linear equations of 100 supports in 0.01s. Estimate: 0.0min
Solved linear equations for all supports. Elapsed time: 0.04s

After excluding supports with no solutions, 7 fundamental models are left.
After excluding supports with non-positive solutions, 3 fundamental models are left.

We have found 3 fundamental models.
Found additional 1 fundamental models due to symmetry.

There exist exactly 4 unique fundamental models ✨


## $n = 6$

In [86]:
fundamental_models = find_fundamental_models(5, [5])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=5: 931 supports to check
Solved linear equations of 100 supports in 0.06s. Estimate: 0.01min
Solved linear equations for all supports. Elapsed time: 0.17s

After excluding supports with no solutions, 3732 fundamental models are left.
After excluding supports with non-positive solutions, 484 fundamental models are left.

We have found 484 fundamental models.
Found additional 469 fundamental models due to symmetry.

There exist exactly 602 fundamental models ✨


In [87]:
fundamental_models = find_fundamental_models(5, [6])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=6: 3390 supports to check
Solved linear equations of 100 supports in 0.03s. Estimate: 0.02min
Solved linear equations for all supports. Elapsed time: 0.39s

After excluding supports with no solutions, 1260 fundamental models are left.
After excluding supports with non-positive solutions, 172 fundamental models are left.

We have found 172 fundamental models.
Found additional 164 fundamental models due to symmetry.

There exist exactly 254 fundamental models ✨


In [89]:
fundamental_models = find_fundamental_models(5, [7])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=7: 8455 supports to check
Solved linear equations of 100 supports in 0.03s. Estimate: 0.04min
Solved linear equations for all supports. Elapsed time: 0.71s

After excluding supports with no solutions, 1014 fundamental models are left.
After excluding supports with non-positive solutions, 59 fundamental models are left.

We have found 59 fundamental models.
Found additional 52 fundamental models due to symmetry.

There exist exactly 88 fundamental models ✨


In [93]:
fundamental_models = find_fundamental_models(5, [8])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=8: 16643 supports to check
Solved linear equations of 100 supports in 0.03s. Estimate: 0.09min
Solved linear equations for all supports. Elapsed time: 1.15s

After excluding supports with no solutions, 548 fundamental models are left.
After excluding supports with non-positive solutions, 16 fundamental models are left.

We have found 16 fundamental models.
Found additional 16 fundamental models due to symmetry.

There exist exactly 24 fundamental models ✨


In [92]:
fundamental_models = find_fundamental_models(5, [9])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print()
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=9: 29114 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 0.12min
Solved linear equations for all supports. Elapsed time: 1.85s

After excluding supports with no solutions, 574 fundamental models are left.
After excluding supports with non-positive solutions, 2 fundamental models are left.

We have found 2 fundamental models.
Found additional 2 fundamental models due to symmetry.

There exist exactly 2 fundamental models ✨


## $n = 7$

### $d = 3,4,5$

In [94]:
fundamental_models = find_fundamental_models(6, [3,4,5])
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=3: 16 supports to check
Solved linear equations for all supports. Elapsed time: 0.03s

d=4: 81 supports to check
Solved linear equations for all supports. Elapsed time: 0.15s

d=5: 931 supports to check
Solved linear equations of 100 supports in 0.14s. Estimate: 0.02min
Solved linear equations for all supports. Elapsed time: 1.08s

After excluding supports with no solutions, 0 fundamental models are left.
After excluding supports with non-positive solutions, 0 fundamental models are left.

We have found 0 fundamental models.
Found additional 0 fundamental models due to symmetry.

There exist exactly 0 fundamental models ✨


### $d = 6$

In [123]:
n = 6
d = 6
fundamental_models = find_fundamental_models_core(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=6: 8125 supports to check
Solved linear equations for all supports. Elapsed time: 14.04s

After excluding supports with no solutions, 124194 fundamental models are left.
After excluding supports with non-positive solutions, 8387 fundamental models are left.

We have found 8387 fundamental models.
Found additional 8327 fundamental models due to symmetry.

There exist exactly 6710 unique fundamental models ✨


### $d = 7$

In [127]:
n = 6
d = 7
fundamental_models = find_fundamental_models_core(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=7: 43762 supports to check
Solved linear equations for all supports. Elapsed time: 11.86s

After excluding supports with no solutions, 59293 fundamental models are left.
After excluding supports with non-positive solutions, 2403 fundamental models are left.

We have found 2403 fundamental models.
Found additional 2252 fundamental models due to symmetry.

There exist exactly 2421 unique fundamental models ✨


### $d = 8$

In [132]:
n = 6
d = 8
fundamental_models = find_fundamental_models_core(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=8: 147585 supports to check
Solved linear equations for all supports. Elapsed time: 18.66s

After excluding supports with no solutions, 49432 fundamental models are left.
After excluding supports with non-positive solutions, 524 fundamental models are left.

We have found 524 fundamental models.
Found additional 513 fundamental models due to symmetry.

There exist exactly 643 unique fundamental models ✨


### $d = 9$

In [134]:
n = 6
d = 9
fundamental_models = find_fundamental_models_core(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=9: 411099 supports to check
Solved linear equations for all supports. Elapsed time: 25.66s

After excluding supports with no solutions, 68892 fundamental models are left.
After excluding supports with non-positive solutions, 177 fundamental models are left.

We have found 177 fundamental models.
Found additional 160 fundamental models due to symmetry.

There exist exactly 198 unique fundamental models ✨


### $d = 10$

In [98]:
n = 6
d = 10
fundamental_models = find_fundamental_models(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=10: 936076 supports to check
Solved linear equations of 100 supports in 0.0s. Estimate: 0.66min
Solved linear equations for all supports. Elapsed time: 205.18s

After excluding supports with no solutions, 92590 fundamental models are left.
After excluding supports with non-positive solutions, 30 fundamental models are left.

We have found 30 fundamental models.
Found additional 30 fundamental models due to symmetry.
Found 32 potential fundamental models


### $d = 11$

In [99]:
n = 6
d = 11
fundamental_models = find_fundamental_models(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=11: 1933243 supports to check
Solved linear equations of 100 supports in 0.02s. Estimate: 5.27min
Solved linear equations for all supports. Elapsed time: 387.76s

After excluding supports with no solutions, 131297 fundamental models are left.
After excluding supports with non-positive solutions, 4 fundamental models are left.

We have found 4 fundamental models.
Found additional 4 fundamental models due to symmetry.
Found 4 potential fundamental models


## $n = 8$

In [136]:
n = 7
d = 7
fundamental_models = find_fundamental_models(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=7: 76922 supports to check
Solved linear equations of 100 supports in 0.3s. Estimate: 3.84min
Solved linear equations for all supports. Elapsed time: 272.91s

After excluding supports with no solutions, 4837923 fundamental models are left.
After excluding supports with non-positive solutions, 152670 fundamental models are left.

We have found 152670 fundamental models.
Found additional 152467 fundamental models due to symmetry.

There exist exactly 83906 unique fundamental models ✨


In [103]:
n = 7
d = 8
fundamental_models = find_fundamental_models(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=8: 436896 supports to check
Solved linear equations of 100 supports in 0.13s. Estimate: 9.29min
Solved linear equations for all supports. Elapsed time: 946.68s

After excluding supports with no solutions, 2198262 fundamental models are left.
After excluding supports with non-positive solutions, 28613 fundamental models are left.

We have found 28613 fundamental models.
Found additional 28419 fundamental models due to symmetry.

There exist exactly 23285 unique fundamental models ✨


In [104]:
n = 7
d = 9
fundamental_models = find_fundamental_models(n, d)
fundamental_models = set([tuple(x) for x,y in fundamental_models])
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=9: 1942505 supports to check
Solved linear equations of 100 supports in 0.24s. Estimate: 76.81min
Solved linear equations for all supports. Elapsed time: 2970.58s

After excluding supports with no solutions, 3444306 fundamental models are left.
After excluding supports with non-positive solutions, 8196 fundamental models are left.

We have found 8196 fundamental models.
Found additional 7901 fundamental models due to symmetry.

There exist exactly 6445 unique fundamental models ✨


In [137]:
n = 7
d = 10
fundamental_models = find_fundamental_models_core(n, d)
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")


d=10: 6431744 supports to check
Solved linear equations for all supports. Elapsed time: 4905.01s

After excluding supports with no solutions, 5894306 fundamental models are left.
After excluding supports with non-positive solutions, 1525 fundamental models are left.

We have found 1525 fundamental models.
Found additional 1495 fundamental models due to symmetry.

There exist exactly 3020 unique fundamental models ✨


In [ ]:
n = 7
d = 11
fundamental_models = find_fundamental_models(n, d)
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")

In [ ]:
n = 7
d = 12
fundamental_models = find_fundamental_models(n, d)
print("")
print(f"There exist exactly {len(fundamental_models)} unique fundamental models ✨")